# Notebook para el filtrado de datos segun distancia estaciones en "data/raw/aruba_stations.json"

In [5]:
import pandas as pd
import numpy as np

json_data_stations = pd.read_json("/home/igomez/WSPython/Proyecto-Aruba-SuperSexyBoys/data/raw/aruba_stations.json")

target_lat = 12.473972584225828
target_lon = -69.89153669806423

def calculate_haversine_distance_meters(latitude_origin, longitude_origin, latitude_destination, longitude_destination):
    # Mean radius of the Earth in meters
    earth_radius_meters = 6371000.0

    # Convert all coordinates from degrees to radians
    lat_origin_rad, lon_origin_rad, lat_dest_rad, lon_dest_rad = map(
        np.radians,
        [latitude_origin, longitude_origin, latitude_destination, longitude_destination]
    )

    # Calculate the difference in coordinates
    delta_latitude = lat_dest_rad - lat_origin_rad
    delta_longitude = lon_dest_rad - lon_origin_rad

    # Calculate the square of half the chord length between the points (Haversine formula)
    half_chord_length_squared = np.sin(delta_latitude / 2.0)**2 + \
                                np.cos(lat_origin_rad) * np.cos(lat_dest_rad) * np.sin(delta_longitude / 2.0)**2

    # Calculate the angular distance in radians
    angular_distance_radians = 2 * np.arcsin(np.sqrt(half_chord_length_squared))

    # Return the physical distance in meters
    return earth_radius_meters * angular_distance_radians

json_data_stations_ordered = json_data_stations.copy()

json_data_stations_ordered['distance_m'] = calculate_haversine_distance_meters(
    json_data_stations_ordered['latitude'],
    json_data_stations_ordered['longitude'],
    target_lat,
    target_lon
)

json_data_stations_ordered = json_data_stations_ordered.sort_values(by="distance_m", ascending=True).reset_index(drop=True)
json_data_stations_ordered = json_data_stations_ordered.head(3)

json_data_stations_ordered.to_json("/home/igomez/WSPython/Proyecto-Aruba-SuperSexyBoys/data/processed/aruba_stations_ordered_by_distance_m.json")